# 04. Predictive Machine Learning Modeling: Loan Recovery
## Project: AI-Driven Loan Recovery & Risk Analytics

### Overview
This notebook trains, evaluates, and compares classification machine learning models to predict the probability of successful loan recovery:
1. **Preprocessing Pipeline**: Handling numerical scaling and categorical One-Hot Encoding via `ColumnTransformer`.
2. **Model Training & Comparison**:
   - Logistic Regression (Class Weighted)
   - Random Forest Classifier
   - XGBoost Classifier
   - LightGBM Classifier (Champion Model)
3. **Model Evaluation**: ROC-AUC, PR-AUC, Accuracy, Precision, Recall, F1-Score, and Confusion Matrix.
4. **Feature Importance & Interpretability**: Identifying key drivers of debt recovery.
5. **Model Serialization**: Saving champion pipeline to `models/recovery_predictor_lightgbm.pkl`.


In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, roc_curve

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import lightgbm as lgb

df = pd.read_csv("../data/processed/loan_recovery_master.csv")
delinquent_df = df[df["overdue_days"] > 30].copy()
delinquent_df["recovery_target"] = delinquent_df["recovery_status"].apply(lambda s: 1 if s in ["Fully Recovered", "Partially Recovered"] else 0)

print(f"Sample Size: {delinquent_df.shape}")
print(delinquent_df["recovery_target"].value_counts(normalize=True))


### 1. Building Preprocessing Pipeline & Splitting Data

In [ ]:
numeric_features = [
    "credit_score", "annual_income", "loan_amount", "interest_rate",
    "loan_term_months", "overdue_days", "dti_ratio", "ltv_ratio",
    "is_secured", "contact_attempts", "promise_to_pay_kept",
    "settlement_discount_pct", "delinquency_severity_score"
]

categorical_features = [
    "loan_type", "region", "city_tier", "employment_status",
    "housing_status", "collateral_type", "origination_channel",
    "primary_channel"
]

X = delinquent_df[numeric_features + categorical_features]
y = delinquent_df["recovery_target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features)
    ]
)
print("Pipeline preprocessor configured.")


### 2. Model Training & Evaluation

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=150, max_depth=8, random_state=42),
    "XGBoost": xgb.XGBClassifier(n_estimators=150, max_depth=5, learning_rate=0.05, random_state=42),
    "LightGBM": lgb.LGBMClassifier(n_estimators=150, max_depth=5, learning_rate=0.05, random_state=42, verbose=-1)
}

results = []
plt.figure(figsize=(9, 6))

for name, clf in models.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("classifier", clf)])
    pipe.fit(X_train, y_train)
    
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba)
    })
    
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, y_proba):.3f})")

plt.plot([0, 1], [0, 1], 'k--')
plt.title("Comparative ROC-AUC Benchmark", fontweight="bold")
plt.legend(loc="lower right")
plt.show()

res_df = pd.DataFrame(results).set_index("Model")
display(res_df)
